# GWO Feature Selection with SMOTE Group K-Fold CV

This notebook demonstrates:
1. **Grey Wolf Optimizer (GWO)** for feature selection
2. **SMOTE** for handling class imbalance
3. **Group K-Fold Cross-Validation** to prevent data leakage from same patients

## Key Concepts

- **GWO**: A metaheuristic optimization algorithm that mimics the hunting behavior of grey wolves to find optimal feature subsets
- **SMOTE**: Synthetic Minority Over-sampling Technique creates synthetic samples to balance class distribution
- **Group K-Fold**: Ensures all data from the same patient stays in the same fold
- **Binary Classification**: Predicting readmission within 30 days (1) vs. not readmitted within 30 days (0)

## Additional Features Included

This analysis includes additional features not dropped:
- `patient_nbr`: For group-based cross-validation
- `admission_type_id`: Type of admission (one-hot encoded)
- `discharge_disposition_id`: Where patient was discharged (one-hot encoded)
- `admission_source_id`: Source of admission (one-hot encoded)
- `A1Cresult`: A1C test result (ordinal encoded: None < Norm < >7 < >8)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Import custom modules
import sys
sys.path.insert(0, '../')

from src import config
from src.data_processing import preprocess_pipeline
from src.feature_selection import select_features_gwo, GWOFeatureSelector
from src.cross_validation import evaluate_with_smote_group_kfold, SMOTEGroupKFoldCV

# Set random seed
np.random.seed(config.RANDOM_STATE)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Data Preprocessing with Patient Groups

We preserve patient IDs to enable group-based cross-validation.

In [ ]:
# Load and preprocess data (keeping patient groups)
result = preprocess_pipeline(
    filepath=None,
    random_state=config.RANDOM_STATE,
    keep_groups=True,
)

# Unpack results
X_train, X_val, X_test, Y_train, Y_val, Y_test, groups_train, groups_val, groups_test, raw_df = result

In [ ]:
# Examine class distribution
print("Class Distribution in Training Set:")
print(Y_train.value_counts())
print(f"\nClass Imbalance Ratio: {Y_train.value_counts()[0] / Y_train.value_counts()[1]:.2f}:1")

# Visualize class distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

Y_train.value_counts().plot(kind='bar', ax=ax[0])
ax[0].set_title('Class Distribution (Training Set)')
ax[0].set_xlabel('Class')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['No Readmission', 'Readmission'], rotation=0)

Y_train.value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%')
ax[1].set_title('Class Distribution (%)')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\nNumber of features: {X_train.shape[1]}")
print(f"Number of unique patients in training: {groups_train.nunique()}")

## 2. Feature Selection using Grey Wolf Optimizer (GWO)

GWO is a nature-inspired metaheuristic algorithm that optimizes feature selection by:
- Treating each feature subset as a position in search space
- Using a population of wolves that hunt (search) for optimal solutions
- Evaluating fitness based on cross-validation performance

In [ ]:
# Initialize base estimator
base_estimator = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
)

print(f"Original number of features: {X_train.shape[1]}")
print("\nStarting GWO feature selection...")
print("(This may take several minutes)")

In [ ]:
# Perform GWO feature selection
gwo_selector, selected_features = select_features_gwo(
    X_train=X_train,
    y_train=Y_train,
    estimator=base_estimator,
    n_wolves=15,  # Population size
    n_iterations=25,  # Number of iterations
    cv_folds=3,  # CV folds for fitness evaluation
    threshold=0.5,  # Binary selection threshold
    random_state=config.RANDOM_STATE,
)

In [ ]:
# Display selected features
print(f"\nSelected {len(selected_features)} out of {X_train.shape[1]} features:")
print("\nSelected Features:")
for i, feature in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feature}")

print(f"\nFeature reduction: {(1 - len(selected_features)/X_train.shape[1])*100:.1f}%")
print(f"Best F1 Score from GWO: {gwo_selector.best_score_:.4f}")

In [ ]:
# Transform datasets using selected features
X_train_selected = X_train[selected_features]
X_val_selected = X_val[selected_features]
X_test_selected = X_test[selected_features]

print(f"Transformed training set shape: {X_train_selected.shape}")

## 3. Model Evaluation with SMOTE Group K-Fold CV



1. **SMOTE**: Addresses class imbalance by creating synthetic minority samples
2. **Group K-Fold**: Prevents data leakage by keeping all encounters from the same patient in the same fold
3. **Applied within CV**: SMOTE is applied only on training folds to avoid leakage

In [ ]:
# Combine train and validation sets for cross-validation
X_train_val = pd.concat([X_train_selected, X_val_selected], axis=0)
Y_train_val = pd.concat([Y_train, Y_val], axis=0)
groups_train_val = pd.concat([groups_train, groups_val], axis=0)

print(f"Combined train+val size: {X_train_val.shape}")
print(f"Number of unique patients: {groups_train_val.nunique()}")
print(f"Class distribution: {Y_train_val.value_counts().to_dict()}")

In [ ]:
# Check for NaN values in the data
print("Checking for NaN values...")
print(f"X_train_val NaNs: {X_train_val.isnull().sum().sum()}")
print(f"Y_train_val NaNs: {Y_train_val.isnull().sum()}")

if X_train_val.isnull().sum().sum() > 0:
    print("\nColumns with NaN values:")
    nan_cols = X_train_val.columns[X_train_val.isnull().any()].tolist()
    for col in nan_cols:
        print(f"  {col}: {X_train_val[col].isnull().sum()} NaNs")
    
    print("\nSample of data types:")
    print(X_train_val.dtypes.value_counts())

### 3a. Random Forest with SMOTE Group K-Fold CV

In [ ]:
# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=5,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
)

# Evaluate with SMOTE Group K-Fold
rf_metrics, rf_fold_scores = evaluate_with_smote_group_kfold(
    X=X_train_val,
    y=Y_train_val,
    groups=groups_train_val,
    estimator=rf_model,
    n_splits=5,
    smote_sampling_strategy=0.5,
    smote_k_neighbors=10,
    random_state=config.RANDOM_STATE,
    verbose=True,
)

In [ ]:
# Visualize fold-wise performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
for ax, metric in zip(axes.flat, metrics_to_plot):
    rf_fold_scores.plot(x='fold', y=metric, kind='bar', ax=ax, legend=False)
    ax.set_title(f'Random Forest - {metric.upper()} per Fold')
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric.upper())
    ax.axhline(y=rf_metrics[f'{metric}_mean'], color='r', linestyle='--', label='Mean')
    ax.legend()
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

### 3b. XGBoost with SMOTE Group K-Fold CV

In [ ]:
# XGBoost model
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss",
)

# Evaluate with SMOTE Group K-Fold
xgb_metrics, xgb_fold_scores = evaluate_with_smote_group_kfold(
    X=X_train_val,
    y=Y_train_val,
    groups=groups_train_val,
    estimator=xgb_model,
    n_splits=5,
    smote_sampling_strategy="auto",
    smote_k_neighbors=5,
    random_state=config.RANDOM_STATE,
    verbose=True,
)

In [ ]:
# Visualize fold-wise performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
for ax, metric in zip(axes.flat, metrics_to_plot):
    xgb_fold_scores.plot(x='fold', y=metric, kind='bar', ax=ax, legend=False, color='orange')
    ax.set_title(f'XGBoost - {metric.upper()} per Fold')
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric.upper())
    ax.axhline(y=xgb_metrics[f'{metric}_mean'], color='r', linestyle='--', label='Mean')
    ax.legend()
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

### 3c. Compare Model Performance

In [ ]:
# Compare models
comparison_data = {
    'Model': ['Random Forest', 'XGBoost'],
    'Accuracy': [rf_metrics['accuracy_mean'], xgb_metrics['accuracy_mean']],
    'Precision': [rf_metrics['precision_mean'], xgb_metrics['precision_mean']],
    'Recall': [rf_metrics['recall_mean'], xgb_metrics['recall_mean']],
    'F1 Score': [rf_metrics['f1_mean'], xgb_metrics['f1_mean']],
}

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison (Mean CV Scores):")
print(comparison_df.to_string(index=False))

# Visualize comparison
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1 Score']].plot(
    kind='bar', figsize=(10, 6), rot=0
)
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Final Model Training and Test Evaluation

In [ ]:
# Select best model
rf_f1 = rf_metrics['f1_mean']
xgb_f1 = xgb_metrics['f1_mean']

if rf_f1 > xgb_f1:
    print(f"Random Forest performed better (F1: {rf_f1:.4f} vs {xgb_f1:.4f})")
    best_model = rf_model
    best_name = "Random Forest"
else:
    print(f"XGBoost performed better (F1: {xgb_f1:.4f} vs {rf_f1:.4f})")
    best_model = xgb_model
    best_name = "XGBoost"

# Apply SMOTE to full training data
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy="auto",
    k_neighbors=5,
    random_state=config.RANDOM_STATE,
)
X_train_val_resampled, Y_train_val_resampled = smote.fit_resample(X_train_val, Y_train_val)

print(f"\nTraining final {best_name} model...")
print(f"Original samples: {len(X_train_val)}")
print(f"Resampled samples: {len(X_train_val_resampled)}")

# Train final model
final_model = best_model.__class__(**best_model.get_params())
final_model.fit(X_train_val_resampled, Y_train_val_resampled)

print("Training complete!")

In [ ]:
# Evaluate on test set
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred = final_model.predict(X_test_selected)
y_test_proba = final_model.predict_proba(X_test_selected)[:, 1]

print("\n" + "="*70)
print("Test Set Performance")
print("="*70)
print("\nClassification Report:")
print(classification_report(Y_test, y_test_pred, target_names=["No Readmission", "Readmission"]))

In [ ]:
# Visualize confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    Y_test, y_test_pred,
    display_labels=["No Readmission", "Readmission"],
    cmap='Blues',
    ax=ax
)
ax.set_title(f'{best_name} - Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(Y_test, y_test_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'{best_name} - ROC Curve (Test Set)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Summary and Insights

In [ ]:
from sklearn.metrics import f1_score

print("="*70)
print("SUMMARY")
print("="*70)

print(f"\n1. Feature Selection (GWO):")
print(f"   - Original features: {X_train.shape[1]}")
print(f"   - Selected features: {len(selected_features)}")
print(f"   - Feature reduction: {(1 - len(selected_features)/X_train.shape[1])*100:.1f}%")
print(f"   - GWO best CV F1: {gwo_selector.best_score_:.4f}")

print(f"\n2. Cross-Validation Results (5-Fold Group CV with SMOTE):")
print(f"   - Random Forest F1: {rf_f1:.4f} (+/- {rf_metrics['f1_std']:.4f})")
print(f"   - XGBoost F1:       {xgb_f1:.4f} (+/- {xgb_metrics['f1_std']:.4f})")

test_f1 = f1_score(Y_test, y_test_pred)
print(f"\n3. Test Set Performance ({best_name}):")
print(f"   - F1 Score: {test_f1:.4f}")
print(f"   - ROC AUC:  {roc_auc:.4f}")

print(f"\n4. Class Imbalance Handling:")
print(f"   - Original imbalance: {Y_train.value_counts()[0] / Y_train.value_counts()[1]:.2f}:1")
print(f"   - SMOTE applied within each CV fold")
print(f"   - Group K-Fold prevented patient data leakage")

print("\n" + "="*70)

## Conclusions

This notebook demonstrated:

1. **GWO Feature Selection**: Reduced feature space while maintaining model performance
2. **SMOTE**: Effectively addressed class imbalance in training data
3. **Group K-Fold CV**: Ensured robust evaluation by preventing data leakage from same patients

### Key Takeaways:

- Feature selection reduced dimensionality while improving or maintaining performance
- SMOTE helped balance the training data without compromising validation integrity
- Group-based cross-validation provided realistic performance estimates
- The combination of these techniques offers a robust approach to imbalanced medical prediction tasks